# Lab 1 · Exercise A — Gaussian, toy data, fits

**Goals:** build a pdf, generate toy data, fit by maximum likelihood (unbinned and binned), plot.
Fill every cell marked `TODO`, run the notebook top to bottom, and answer the three questions at the end.
Reference: RooFit tutorials `rf101_basics`, `rf201`, `rf202`.

In [1]:
import ROOT
%jsroot on 
import numpy as np

MATRICOLA = 1263603
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## A1 · Unbinned fit of a Gaussian
Gaussian with mean = 0 and σ = 1. Plot it. Change σ to 3. Generate 10 000 **unbinned** events, fit by ML, plot.

In [2]:
x     = ROOT.RooRealVar("x", "x", -10, 10)
mean  = ROOT.RooRealVar("mean", "mean", 0, -10, 10)
sigma = ROOT.RooRealVar("sigma", "sigma", 1, 0.1, 10)    # start with sigma = 1

gauss = ROOT.RooGaussian("gauss", "gauss", x, mean, sigma)

frame0 = x.frame(Title="Gaussian pdf: sigma = 1 and sigma = 3")
gauss.plotOn(frame0, LineColor=ROOT.kBlue)     # sigma = 1
sigma.setVal(3)
gauss.plotOn(frame0, LineColor=ROOT.kRed)      # sigma = 3
c0 = ROOT.TCanvas("c0", "c0", 800, 500)
frame0.Draw()
c0.Draw()

In [3]:
# sigma is now 3: generate and fit
data   = gauss.generate({x}, 10000)
result = gauss.fitTo(data, Save=True, PrintLevel=-1)

frame1 = x.frame(Title="Unbinned ML fit")
data.plotOn(frame1)
gauss.plotOn(frame1)
c1 = ROOT.TCanvas("c1", "c1", 800, 500)
frame1.Draw()
c1.Draw()
mean.Print(); sigma.Print()

RooRealVar::mean = 0.00806151 +/- 0.0301798  L(-10 - 10) 
RooRealVar::sigma = 3.0023 +/- 0.0219288  L(0.1 - 10) 


In [4]:
check("pdf and data exist", gauss is not None and data is not None and result is not None)
check("fit converged", result.status() == 0)
check("sigma compatible with truth (3)", abs(sigma.getVal() - 3) < 4 * sigma.getError())
check("mean compatible with truth (0)", abs(mean.getVal() - 0) < 4 * mean.getError())

PASS  pdf and data exist
PASS  fit converged
PASS  sigma compatible with truth (3)
PASS  mean compatible with truth (0)


## A2 · Binned fit of an exponential
Exponential pdf with rate = −1/τ, τ = 4.5 (mean life). Generate a **binned** dataset of 1000 events,
bin width 0.5. Fit by ML and plot.
*Tip:* express the rate with a `RooFormulaVar("rate", "-1./@0", ROOT.RooArgList(tau))`;
set the number of bins with `t.setBins(...)` **before** `generateBinned`.

In [5]:
t    = ROOT.RooRealVar("t", "t", 0, 10)
tau  = ROOT.RooRealVar("tau", "tau", 4.5, 0.5, 10)
rate = ROOT.RooFormulaVar("rate", "-1./@0", ROOT.RooArgList(tau))
expo = ROOT.RooExponential("expo", "expo", t, rate)

t.setBins(20)              # bin width 0.5
data_b = expo.generateBinned({t}, 1000)
# start the fit from a wrong value, so the fit has some work to do:
tau.setVal(2.0)
res_b = expo.fitTo(data_b, Save=True, PrintLevel=-1)

frame2 = t.frame(Title="Binned ML fit")
data_b.plotOn(frame2)
expo.plotOn(frame2)
c2 = ROOT.TCanvas("c2", "c2", 800, 500)
frame2.Draw()
c2.Draw()
tau.Print()

RooRealVar::tau = 4.95944 +/- 0.295975  L(0.5 - 10) 


In [6]:
check("binned dataset built", data_b is not None and res_b is not None)
check("fit converged", res_b.status() == 0)
check("tau compatible with truth (4.5)", abs(tau.getVal() - 4.5) < 4 * tau.getError())

PASS  binned dataset built
PASS  fit converged
PASS  tau compatible with truth (4.5)


## A3 · (optional) Ranges of a parameter and of an observable
Refit the Gaussian of A1 in two modified settings and compare **best values and errors** with A1:
(a) a **parameter limit** that excludes the true value, `sigma.setRange(0.1, 2)`; (b) a **sub-range of the observable**, `x.setRange("fit", -5, 5)` and `fitTo(..., Range="fit")`.
Look at the printout of (a): where did sigma stop, and what is its error?

In [7]:
# A3 (a): a parameter limit that excludes the true value (sigma = 3)
ref = (mean.getVal(), mean.getError(), sigma.getVal(), sigma.getError())          # best values of A1
print("A1            : mean = %.3f +- %.3f   sigma = %.3f +- %.3f" % ref)
sigma.setRange(0.1, 2); sigma.setVal(1); mean.setVal(0)
res_a = gauss.fitTo(data, Save=True, PrintLevel=-1)
print("sigma < 2     : mean = %.3f +- %.3f   sigma = %.3f +- %.3f" % (mean.getVal(), mean.getError(), sigma.getVal(), sigma.getError()))

# A3 (b): fit only a sub-range of the observable x
sigma.setRange(0.1, 10); sigma.setVal(1); mean.setVal(0)
x.setRange("fit", -5, 5)
res_b = gauss.fitTo(data, Range="fit", Save=True, PrintLevel=-1)
print("x in [-5, 5]  : mean = %.3f +- %.3f   sigma = %.3f +- %.3f" % (mean.getVal(), mean.getError(), sigma.getVal(), sigma.getError()))

A1            : mean = 0.008 +- 0.030   sigma = 3.002 +- 0.022
sigma < 2     : mean = 0.008 +- 0.020   sigma = 2.000 +- 0.000
x in [-5, 5]  : mean = 0.024 +- 0.039   sigma = 2.953 +- 0.044


## Questions (answer in this cell, 2–3 lines each)

1. Are the fitted values and errors compatible with the true ones? How did you decide?
- Yes! the pulls (fit − true)/error are 0.27σ (mean), 0.11σ (σ) and 1.55σ (τ), all below 2σ. The errors also match the expected σ/√N = 0.030 and σ/√(2N) = 0.021.
2. Repeat A1 with 100 events. What happens to the errors? Is the scaling what you expect?
- With 100 events the errors are ~9.7× larger (mean −0.249 ± 0.294, σ 2.927 ± 0.212), and the values are still compatible with the truth (0.85σ and 0.34σ). This matches the expected 1/√N scaling: √(10000/100) = 10.
3. Why is the error on σ different from the error on the mean?
- The Fisher information is N/σ² for the mean and 2N/σ² for σ, so σ_μ = σ/√N and σ_σ = σ/√(2N): σ is measured √2 better.
4. (A3, optional) What changed in the best values and in the errors in the two cases, and why?
- (a) σ sticks at the limit (2.000 ± 0.000, a meaningless error), and the mean error falls to 2/√N = 0.020 because it comes from the wrong model. (b) The pdf is renormalised in [−5, 5], so there is no bias; but the tails are lost, so σ's error doubles and the mean's grows ~30%.

In [8]:
# Question 2: repeat A1 with only 100 events and compare the errors
sigma.setVal(3); mean.setVal(0)
data_small = gauss.generate({x}, 100)
res_small  = gauss.fitTo(data_small, Save=True, PrintLevel=-1)

print("N = 10000 : mean = %.3f +- %.3f   sigma = %.3f +- %.3f" % ref)
print("N =   100 : mean = %.3f +- %.3f   sigma = %.3f +- %.3f" % (mean.getVal(), mean.getError(), sigma.getVal(), sigma.getError()))
print("error ratio (100 / 10000): mean %.1f   sigma %.1f   (expected sqrt(10000/100) = 10)"
      % (mean.getError() / ref[1], sigma.getError() / ref[3]))

N = 10000 : mean = 0.008 +- 0.030   sigma = 3.002 +- 0.022
N =   100 : mean = -0.249 +- 0.294   sigma = 2.927 +- 0.212
error ratio (100 / 10000): mean 9.7   sigma 9.7   (expected sqrt(10000/100) = 10)
